 Fine-Tuning a Transformer (GELU)

In [7]:
!pip install transformers

In [8]:
%pip install transformers datasets

Note: you may need to restart the kernel to use updated packages.


In [9]:
# CELL 1 — Imports

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [11]:
# CELL 2 — Load dataset (SST-2 sentiment classification, part of GLUE)
raw_datasets = load_dataset("nyu-mll/glue", "sst2")
print(raw_datasets)
print("Example:", raw_datasets["train"][0])

c:\Users\chari\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\chari\.cache\huggingface\hub\datasets--nyu-mll--glue. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating test split: 100%|██████████| 1821/1821 [00:00<00:00, 76296.64 examples/s]


DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 872
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1821
    })
})
Example: {'sentence': 'hide new secretions from the parental units ', 'label': 0, 'idx': 0}


In [12]:
# CELL 3 — Load tokenizer and model (BERT — uses GELU in its feed-forward layers)
MODEL_NAME = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.to(device)

# Quick check: confirm GELU is the activation inside BERT's feed-forward blocks
print(model.config.hidden_act)  # should print 'gelu'

c:\Users\chari\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\chari\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2605.90it/s]
[transformers] BertForSequen

gelu


In [13]:
# CELL 4 — Tokenize dataset
def tokenize_function(examples):
    return tokenizer(examples["sentence"], truncation=True, max_length=128)

tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
tokenized_datasets = tokenized_datasets.remove_columns(["sentence", "idx"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")

# Use manageable subsets for quick fine-tuning
train_subset = tokenized_datasets["train"].shuffle(seed=42).select(range(4000))
eval_subset = tokenized_datasets["validation"].shuffle(seed=42).select(range(500))

print(train_subset[0])

Map: 100%|██████████| 1821/1821 [00:00<00:00, 15787.65 examples/s]


{'labels': tensor(1), 'input_ids': tensor([  101, 12555,  1010, 11951,  1999, 22092,  2066,  2137, 11345,  1998,
         2757,  1011,  2006,  1999,  2602,  1010,   102]), 'token_type_ids': tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]), 'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1])}


In [14]:
# CELL 5 — DataLoaders with dynamic padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

train_loader = DataLoader(
    train_subset, batch_size=16, shuffle=True, collate_fn=data_collator
)
eval_loader = DataLoader(
    eval_subset, batch_size=32, shuffle=False, collate_fn=data_collator
)

print(f"Train batches: {len(train_loader)} | Eval batches: {len(eval_loader)}")

Train batches: 250 | Eval batches: 16


In [15]:
# CELL 6 — Optimizer and learning rate scheduler
epochs = 3
optimizer = AdamW(model.parameters(), lr=2e-5)

num_training_steps = epochs * len(train_loader)
lr_scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * num_training_steps),
    num_training_steps=num_training_steps
)

print(f"Total training steps: {num_training_steps}")

Total training steps: 750


In [16]:
# CELL 7 — Training loop
model.train()
for epoch in range(epochs):
    running_loss = 0.0
    correct = 0
    total = 0

    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = model(**batch)          # feed-forward layers inside use GELU
        loss = outputs.loss
        logits = outputs.logits

        loss.backward()
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()

        running_loss += loss.item()
        preds = torch.argmax(logits, dim=-1)
        correct += (preds == batch["labels"]).sum().item()
        total += batch["labels"].size(0)

    avg_loss = running_loss / len(train_loader)
    train_acc = 100 * correct / total
    print(f"Epoch {epoch+1}/{epochs} — Loss: {avg_loss:.4f} | Train Acc: {train_acc:.2f}%")

Epoch 1/3 — Loss: 0.4134 | Train Acc: 79.92%
Epoch 2/3 — Loss: 0.1684 | Train Acc: 93.95%
Epoch 3/3 — Loss: 0.0739 | Train Acc: 97.45%


In [17]:
# CELL 8 — Evaluation
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for batch in eval_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        preds = torch.argmax(outputs.logits, dim=-1)
        correct += (preds == batch["labels"]).sum().item()
        total += batch["labels"].size(0)

print(f"Validation Accuracy: {100 * correct / total:.2f}%")

Validation Accuracy: 89.40%


In [18]:
# CELL 9 — Try it on custom sentences
def predict_sentiment(text, model, tokenizer):
    model.eval()
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)
        pred = torch.argmax(probs, dim=-1).item()
    label = "Positive" if pred == 1 else "Negative"
    confidence = probs[0][pred].item()
    return label, confidence

sample_sentences = [
    "This movie was an absolute masterpiece.",
    "I regret watching this, total waste of time.",
    "It was okay, not great but watchable."
]

for sentence in sample_sentences:
    label, conf = predict_sentiment(sentence, model, tokenizer)
    print(f"Sentence: {sentence}\nPrediction: {label} (confidence: {conf:.2f})\n")

Sentence: This movie was an absolute masterpiece.
Prediction: Positive (confidence: 1.00)

Sentence: I regret watching this, total waste of time.
Prediction: Negative (confidence: 1.00)

Sentence: It was okay, not great but watchable.
Prediction: Positive (confidence: 1.00)



In [19]:
# CELL 10 — Inspect GELU in action (optional deep dive)
# Look at one feed-forward block's activation function directly
ffn_layer = model.bert.encoder.layer[0].intermediate
print("Feed-forward intermediate layer:", ffn_layer)
print("Activation function used:", ffn_layer.intermediate_act_fn)

# Compare GELU vs ReLU on a sample input to see the smoother curve
sample_input = torch.linspace(-3, 3, steps=7)
gelu_output = nn.functional.gelu(sample_input)
relu_output = nn.functional.relu(sample_input)

for x, g, r in zip(sample_input, gelu_output, relu_output):
    print(f"x={x.item():.2f} | GELU={g.item():.4f} | ReLU={r.item():.4f}")

Feed-forward intermediate layer: BertIntermediate(
  (dense): Linear(in_features=768, out_features=3072, bias=True)
  (intermediate_act_fn): GELUActivation()
)
Activation function used: GELUActivation()
x=-3.00 | GELU=-0.0040 | ReLU=0.0000
x=-2.00 | GELU=-0.0455 | ReLU=0.0000
x=-1.00 | GELU=-0.1587 | ReLU=0.0000
x=0.00 | GELU=0.0000 | ReLU=0.0000
x=1.00 | GELU=0.8413 | ReLU=1.0000
x=2.00 | GELU=1.9545 | ReLU=2.0000
x=3.00 | GELU=2.9960 | ReLU=3.0000
